# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [7]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [8]:
get_food_calories('cashew')

TypeError: 'FunctionTool' object is not callable

In [9]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],)

In [10]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories total (banana ~105 kcal + apple ~80 kcal).


Enforce tools use:

In [11]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

Approximately 185 calories total.
- Medium banana: 105 kcal
- Medium apple: 80 kcal

Note: sizes vary; adjust if your fruits are larger or smaller.


In [11]:
@function_tool
def get_food_water_content(food_item: str) -> str:
    """Return approximate water content for common foods, in grams per 100g."""
    water_content_per_100g = {
        "apple": 85.6,
        "almonds": 4.4,
        "banana": 74.9,
        "broccoli": 89.3,
        "cucumber": 95.2,
        "orange": 86.8,
        "tomato": 94.5,
        "watermelon": 91.5,
    }

    food_key = food_item.strip().casefold()
    water_grams = water_content_per_100g.get(food_key)
    if water_grams is None:
        available_foods = ", ".join(water_content_per_100g)
        return f"I don't have water-content data for {food_item}. Try: {available_foods}."

    return (
        f"{food_item.title()}: about {water_grams}g of water per 100g "
        f"(approximately {water_grams}% water)."
    )


water_agent = Agent(
    name="Food Water Content Assistant",
    instructions="""
    You provide concise water-content information for foods.
    Use the get_food_water_content tool for supported foods.
    State values per 100g and do not guess when a food is not in the lookup.
    """,
    tools=[get_food_water_content, get_food_calories],
)

In [12]:
with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are there in banana and whats the water content also?"
    )
    print(result.final_output)

- Calories: about 105 kcal per medium banana (118 g).
- Water content: about 74–75% water by weight (roughly 88–89 g water per 118 g banana).
